In [1]:
import pandas as pd

df = pd.read_csv("C:/Users/Rozina I Shaikh/Downloads/archive (2)/Hyderbad_House_price.csv")

print(df.head())
print(df.shape)
print(df.isnull().sum())

   Unnamed: 0            title    location  price(L)  rate_persqft  \
0           0  3 BHK Apartment    Nizampet    108.00          6000   
1           1  3 BHK Apartment  Bachupally     85.80          5500   
2           2  2 BHK Apartment    Dundigal     55.64          5200   
3           3  2 BHK Apartment    Pocharam     60.48          4999   
4           4  3 BHK Apartment      Kollur    113.00          5999   

   area_insqft     building_status  
0         1805  Under Construction  
1         1560  Under Construction  
2         1070  Under Construction  
3         1210  Under Construction  
4         1900  Under Construction  
(3660, 7)
Unnamed: 0         0
title              0
location           0
price(L)           0
rate_persqft       0
area_insqft        0
building_status    0
dtype: int64


In [11]:
# Remove unnecessary index column
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Rename columns to simpler names
df = df.rename(columns={
    "price(L)": "Price",
    "area_insqft": "Area",
    "location": "Location"
})

# Remove duplicate rows
df = df.drop_duplicates()

print(df.head())
print(df.shape)
print(df.isnull().sum())

             title    Location   Price  rate_persqft  Area     building_status
0  3 BHK Apartment    Nizampet  108.00          6000  1805  Under Construction
1  3 BHK Apartment  Bachupally   85.80          5500  1560  Under Construction
2  2 BHK Apartment    Dundigal   55.64          5200  1070  Under Construction
3  2 BHK Apartment    Pocharam   60.48          4999  1210  Under Construction
4  3 BHK Apartment      Kollur  113.00          5999  1900  Under Construction
(3029, 6)
title              0
Location           0
Price              0
rate_persqft       0
Area               0
building_status    0
dtype: int64


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.base import clone

# Input features
X = df[["title", "Location", "Area", "building_status"]]

# Target: house price in lakhs
y = df["Price"]

# Convert text categories into numbers
preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"),
     ["title", "Location", "building_status"]),
    ("numeric", SimpleImputer(strategy="median"), ["Area"])
])

# Divide data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    )
}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    print("\nModel:", name)
    print("Mean Absolute Error:", round(
        mean_absolute_error(y_test, predictions), 2), "lakhs")
    print("R² Score:", round(r2_score(y_test, predictions), 3))


Model: Linear Regression
Mean Absolute Error: 53.55 lakhs
R² Score: 0.492

Model: Random Forest
Mean Absolute Error: 39.23 lakhs
R² Score: 0.491


In [13]:
# Build and train the final sample model
final_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ))
])

final_model.fit(X_train, y_train)

# Enter sample property details
new_house = pd.DataFrame([{
    "title": "3 BHK Apartment",
    "Location": "Kondapur",
    "Area": 1800,
    "building_status": "Ready to move"
}])

predicted_price = final_model.predict(new_house)[0]

print("Predicted House Price:", round(predicted_price, 2), "lakhs")

Predicted House Price: 91.19 lakhs


In [14]:
print("Price summary:")
print(df["Price"].describe())

print("\nArea summary:")
print(df["Area"].describe())

print("\nProperty types:")
print(df["title"].value_counts().head(10))

print("\nBuilding status:")
print(df["building_status"].value_counts())

print("\nMost common locations:")
print(df["Location"].value_counts().head(10))

Price summary:
count    3029.000000
mean      115.609317
std       212.230341
min         1.320000
25%        26.960000
50%        65.000000
75%       119.000000
max      3600.000000
Name: Price, dtype: float64

Area summary:
count     3029.000000
mean      2080.900957
std       1944.569201
min        118.000000
25%       1300.000000
50%       1642.000000
75%       2160.000000
max      45000.000000
Name: Area, dtype: float64

Property types:
title
Residential Plot           1313
3 BHK Apartment             701
2 BHK Apartment             541
4 BHK Apartment             101
4 BHK Villa                  87
3 BHK Villa                  69
2 BHK Independent House      50
4 BHK Independent House      42
3 BHK Independent House      21
2 BHK Independent Floor      21
Name: count, dtype: int64

Building status:
building_status
New                   1112
Under Construction     962
Ready to move          754
Resale                 201
Name: count, dtype: int64

Most common locations:
Location
M